In [75]:
import re, numpy as np, pandas as pd
from pathlib import Path
RAW, CLEAN = Path("data/raw"), Path("data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)
NULL_TOKENS = ["", "NULL", "null", "N/A", "n/a", "NA", "unknown", "Unknown"]
IST = pd.Timedelta(hours=5, minutes=30)

## Load and profile the raw data

In [76]:
from pathlib import Path
import pandas as pd

FILES = [
    "customers_raw.csv",
    "merchants_raw.csv",
    "device_logs_raw.csv",
    "complaints_raw.csv",
    "transactions_raw.csv",
]

# Find the folder containing all the input CSV files.
BASE = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if all((folder / filename).is_file() for filename in FILES)
    ),
    None,
)

if BASE is None:
    raise FileNotFoundError(
        "Could not find all required CSV files. "
        f"Notebook is running from: {Path.cwd()}\n"
        f"Expected files: {', '.join(FILES)}"
    )

RAW = BASE
CLEAN = BASE / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

NULL_TOKENS = ["", "NULL", "null", "N/A", "n/a", "NA", "unknown", "Unknown"]

def load(name):
    path = RAW / f"{name}_raw.csv"
    return pd.read_csv(
        path,
        dtype=str,
        na_values=NULL_TOKENS,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

raw = {
    name: load(name)
    for name in ["customers", "merchants", "device_logs", "complaints", "transactions"]
}

def profile(df, name):
    print(f"{name}: {len(df):,} rows | exact duplicates: {df.duplicated().sum():,}")
    return pd.DataFrame({
        "nulls": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(),
    })

profile(raw["transactions"], "transactions")

transactions: 62,292 rows | exact duplicates: 1,219


,nulls,null_%,unique
txn_id,0,0.0,61073
timestamp,0,0.0,60097
amount,0,0.0,27619
cust_id,2637,4.2,4812
payer_vpa,0,0.0,12843
payee_vpa,633,1.0,586
merchant_id,564,0.9,304
channel,0,0.0,19
status,0,0.0,21
failure_reason,59276,95.2,6


## Clean customers

In [77]:
cu = raw["customers"].copy()
cu["phone10"] = cu["phone"].str.replace(r"\D","",regex=True).str[-10:]

def fix_name(s):
    if pd.isna(s): return s
    s = re.sub(r"\s+"," ",s.strip())
    if "," in s:
        last, first = [x.strip() for x in s.split(",",1)]; s = f"{first} {last}"
    return s.title()
cu["full_name"] = cu["full_name"].map(fix_name)

def parse_multi(s, formats):                 # try each format, fill the gaps
    out = pd.to_datetime(s, format=formats[0], errors="coerce")
    for f in formats[1:]:
        out = out.fillna(pd.to_datetime(s, format=f, errors="coerce"))
    return out

cu["dob"] = parse_multi(cu["dob"], ["%d-%m-%Y","%Y-%m-%d","%d/%m/%y","%b %d, %Y"])
# TRAP: %y turns 61 into 2061. No customer is born after 2010, so subtract 100 years.
cu["dob"] = cu["dob"].mask(cu["dob"].dt.year > 2010, cu["dob"] - pd.DateOffset(years=100))

CITY_MAP = {"bombay":"Mumbai","mumbai":"Mumbai","delhi":"Delhi","new delhi":"Delhi",
  "bangalore":"Bengaluru","bengaluru":"Bengaluru","blr":"Bengaluru","pune":"Pune","poona":"Pune",
  "hyderabad":"Hyderabad","hyd":"Hyderabad","chennai":"Chennai","madras":"Chennai",
  "kolkata":"Kolkata","calcutta":"Kolkata","jaipur":"Jaipur","ahmedabad":"Ahmedabad",
  "amdavad":"Ahmedabad","lucknow":"Lucknow"}
def std_city(s, keep_unmapped=False):
    m = s.str.strip().str.lower().map(CITY_MAP)
    return m.fillna(s.str.strip().str.title()) if keep_unmapped else m
cu["city"] = std_city(cu["city"])

k = cu["kyc_status"].str.lower().str.strip()
cu["kyc"] = np.select([k.str.contains("full",na=False), k.str.contains("min",na=False)],
                      ["full","min"], "pending")

# entity resolution
cu["open_date"] = parse_multi(cu["account_open_date"], ["%d-%m-%Y","%Y-%m-%d","%d %b %Y"])
cu = cu.sort_values(["phone10","cust_id"])
cu["master_id"] = cu.groupby("phone10")["cust_id"].transform("first")
id_map = dict(zip(cu["cust_id"], cu["master_id"]))     # duplicate ID -> real ID
cust = (cu.groupby("master_id", as_index=False).first()   # first non-null per column
          [["master_id","full_name","phone10","email","dob","city","kyc","open_date","upi_vpa"]]
          .rename(columns={"master_id":"cust_id"}))

## Clean merchants and device logs

In [78]:
# Create a working copy
dl = raw["device_logs"].copy()

# Standardize timestamp and handle mixed timezones
dl["timestamp"] = pd.to_datetime(
    dl["timestamp"],
    errors="coerce",
    utc=True
)

# Clean text columns
for col in ["event_type", "device_id"]:
    dl[col] = (
        dl[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

print("Timestamp conversion completed.")
print("Timestamp type:", dl["timestamp"].dtype)

C:\Users\SHIFA\AppData\Local\Temp\ipykernel_22004\3605858143.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dl["timestamp"] = pd.to_datetime(


Timestamp conversion completed.
Timestamp type: datetime64[us, UTC]


#### text 

In [79]:
for col in ["event_type", "device_id"]:
    dl[col] = (
        dl[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

#### VPN flag

In [80]:
vpn_map = {
    "true": True,
    "yes": True,
    "y": True,
    "1": True,
    "false": False,
    "no": False,
    "n": False,
    "0": False
}

dl["is_vpn"] = (
    dl["is_vpn"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(vpn_map)
    .fillna(False)
    .astype(bool)
)

In [81]:
# Clean OS version text
os_text = (
    dl["os_version"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Extract OS version number
os_version_num = os_text.str.extract(
    r"(\d+(?:\.\d+)?)",
    expand=False
)

# Identify operating system
platform = np.where(
    os_text.str.contains("ios|iphone|ipad", na=False),
    "iOS",
    np.where(
        os_text.str.contains("android|^a\d+", regex=True, na=False),
        "Android",
        pd.NA
    )
)

# Create clean OS field
dl["os_clean"] = (
    pd.Series(platform, index=dl.index, dtype="string")
    .str.cat(os_version_num.astype("string"), sep=" ")
    .where(os_version_num.notna())
)

print(dl[["os_version", "os_clean"]].head(10))

   os_version    os_clean
0  Android 13  Android 13
1   android13  Android 13
2  Android 12  Android 12
3      iOS 16      iOS 16
4         A13  Android 13
5  Android 13  Android 13
6      iOS 16      iOS 16
7  Android 13  Android 13
8         A13  Android 13
9       ios17      iOS 17


<>:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\SHIFA\AppData\Local\Temp\ipykernel_22004\3875945933.py:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  os_text.str.contains("android|^a\d+", regex=True, na=False),


In [82]:
dl["ip_city"] = std_city(
    dl["ip_city"],
    keep_unmapped=True
)

In [83]:
dl["cust_id"] = (
    dl["cust_id"]
    .map(id_map)
    .fillna(dl["cust_id"])
)

In [84]:
print("Device logs cleaned successfully.")
print(f"Rows: {len(dl):,}")
print(f"Columns: {len(dl.columns):,}")
print(f"Missing timestamps: {dl['timestamp'].isna().sum():,}")
print(f"Missing customer IDs: {dl['cust_id'].isna().sum():,}")
print(f"VPN events: {dl['is_vpn'].sum():,}")
print(f"Unique devices: {dl['device_id'].nunique():,}")

Device logs cleaned successfully.
Rows: 30,401
Columns: 11
Missing timestamps: 3,622
Missing customer IDs: 627
VPN events: 300
Unique devices: 5,871


## Clean transactions

In [85]:
tx = raw["transactions"].drop_duplicates()                       
test = tx["txn_id"].str.startswith("TEST-",na=False) | (tx["merchant_id"]=="SANDBOX")
tx = tx[~test].copy()                                         

In [86]:
def parse_ts(s):
    s = s.str.strip()
    epoch = s.where(s.str.fullmatch(r"\d{9,11}",na=False)).astype("float")
    parts = [pd.to_datetime(epoch, unit="s", errors="coerce") + IST,
             pd.to_datetime(s, format="%Y-%m-%dT%H:%M:%SZ", errors="coerce") + IST,
             pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce"),
             pd.to_datetime(s, format="%d/%m/%Y %I:%M %p", errors="coerce"),
             pd.to_datetime(s, format="%d-%b-%y %H:%M", errors="coerce")]
    out = parts[0]
    for p in parts[1:]: out = out.fillna(p)
    return out
tx["timestamp"] = parse_ts(tx["timestamp"])

In [87]:
tx["amount"] = pd.to_numeric(
    tx["amount"].str.replace(r"(?i)rs\.?|inr|₹|,|\s","",regex=True), errors="coerce")
tx["amount_abs"] = tx["amount"].abs()      # reversals are negative in only some rows, so use amount_abs

In [88]:
STATUS_MAP = {"success":"SUCCESS","s":"SUCCESS","00":"SUCCESS","failed":"FAILED","fail":"FAILED",
 "f":"FAILED","failed_timeout":"FAILED","pending":"PENDING","p":"PENDING","reversed":"REVERSED",
 "reversal":"REVERSED","r":"REVERSED","refunded":"REFUNDED","refund":"REFUNDED"}
tx["status"]  = tx["status"].str.strip().str.lower().map(STATUS_MAP)
tx["channel"] = tx["channel"].str.strip().str.upper().str.replace(r"[\s\-]+","_",regex=True)
tx["txn_id"]  = tx["txn_id"].str.strip().str.upper()
for c in ["payer_vpa","payee_vpa","device_id"]:
    tx[c] = tx[c].str.strip().str.lower()
tx["ip_city"] = std_city(tx["ip_city"], keep_unmapped=True)

### missing value 

In [89]:
tx["cust_id"] = tx["cust_id"].str.strip().str.upper().map(id_map).fillna(tx["cust_id"])

# (a) via UPI ID, but ONLY IDs that belong to exactly one customer
vpa_map = (cust.dropna(subset=["upi_vpa"]).drop_duplicates("upi_vpa", keep=False)
               .set_index("upi_vpa")["cust_id"])
tx["cust_id"] = tx["cust_id"].fillna(tx["payer_vpa"].map(vpa_map))

# (b) via a device used by exactly one customer
g = dl.dropna(subset=["device_id","cust_id"]).groupby("device_id")["cust_id"].nunique()
dev_map = (dl[dl["device_id"].isin(g[g==1].index)].drop_duplicates("device_id")
             .set_index("device_id")["cust_id"])
tx["cust_id"] = tx["cust_id"].fillna(tx["device_id"].map(dev_map))

In [90]:
tx["label_reported"] = (tx["fraud_label"].str.strip().str.lower()
                        .map({"1":1,"y":1,"fraud":1,"0":0,"n":0}))   # "?" and blank stay NaN

### drived column 

In [91]:
tx = tx.sort_values("timestamp").reset_index(drop=True)
tx["date"] = tx["timestamp"].dt.normalize()
tx["hour"] = tx["timestamp"].dt.hour
tx["dow"]  = tx["timestamp"].dt.day_name()
tx["is_night"] = tx["hour"].isin([22,23,0,1,2,3,4,5])
tx["amount_band"] = pd.cut(tx["amount_abs"], [0,100,500,2000,10000,np.inf],
                           labels=["<=100","100-500","500-2k","2k-10k",">10k"])

In [97]:
FRAUD_KW = r"otp|sim|unauthor|collect|dubai|never left|bank officer|kyc|beneficiar|seller|shopping page|recogni|lost network|without my knowledge|charges on my card"

co["is_fraud_complaint"] = (
    co["text"]
    .astype("string")
    .str.lower()
    .str.contains(FRAUD_KW, regex=True, na=False)
)

print(co["is_fraud_complaint"].value_counts())

NameError: name 'co' is not defined

In [98]:
FRAUD_KW = r"otp|sim|unauthor|collect|dubai|never left|bank officer|kyc|beneficiar|seller|shopping page|recogni|lost network|without my knowledge|charges on my card"

co["is_fraud_complaint"] = (
    co["text"]
    .astype("string")
    .str.lower()
    .str.contains(FRAUD_KW, regex=True, na=False)
)

print(co["is_fraud_complaint"].value_counts())

NameError: name 'co' is not defined

In [99]:
print("TEST")

TEST
